In [ ]:
# ======================================================
# STEP 2: Import dependencies
# ======================================================
import pandas as pd
import numpy as np
import string
import re
import seaborn as sns
import matplotlib.pyplot as plt

import nltk
# NLTK setup
nltk.download('stopwords')
nltk.download('wordnet')
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

# import fasttext
# from gensim.models import FastText, Word2Vec
from torch import nn
import torch
from torch.utils.data import DataLoader, Dataset


In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import os

# Path to your folder in Google Drive
folder_path = "/content/drive/MyDrive/Mini"

# Load all four CSVs
fake_politifact = pd.read_csv(os.path.join(folder_path, "politifact_fake.csv"))
real_politifact = pd.read_csv(os.path.join(folder_path, "politifact_real.csv"))
fake_gossipcop = pd.read_csv(os.path.join(folder_path, "gossipcop_fake.csv"))
real_gossipcop = pd.read_csv(os.path.join(folder_path, "gossipcop_real.csv"))
welfake = pd.read_csv(os.path.join(folder_path, "WELFake_Dataset.csv"))
news = pd.read_csv(os.path.join(folder_path, "news.csv"))


# Add labels: 0 = fake, 1 = real
fake_politifact["label"] = 0
real_politifact["label"] = 1
fake_gossipcop["label"] = 0
real_gossipcop["label"] = 1

# Merge into one DataFrame
df1 = pd.concat([fake_politifact, real_politifact, fake_gossipcop, real_gossipcop])
df2 = pd.concat([welfake, news])
df1["combined_text"] = df1["title"].astype(str)
df1 = df1[["combined_text", "label"]]

df2["combined_text"] = df2["title"].astype(str) + " " + df2["text"].astype(str)
df2 = df2[["combined_text", "label"]]

df = pd.concat([df1, df2])
# Combine the two DataFrames

# Keep only useful columns
fakenewsnet = df1[["combined_text", "label"]]
welfake["combined_text"] = welfake["title"].astype(str) + " " + welfake["text"].astype(str)
welfake = welfake[["combined_text", "label"]]
news["combined_text"] = news["title"].astype(str) + " " + news["text"].astype(str)
news = news[["combined_text", "label"]]
# Save combined dataset
output_path = os.path.join(folder_path, "FakeNewsNet_clean.csv")
df.to_csv(output_path, index=False)

print("✅ Combined dataset saved at:", output_path)
print("📊 Total samples:", len(df))
print(df.columns)
print(fake_politifact.columns)
print(fake_gossipcop.columns)
print(real_politifact.columns)
print(real_gossipcop.columns)
print(welfake.columns)
print(news.columns)
print(fakenewsnet.columns)


In [ ]:
# Show the structure and a few examples
print(df.head())
print("\nDataset Info:")
print(df.info())

# Check for class balance
print("\nLabel Distribution:")
print(df['label'].value_counts())


In [ ]:
# Check missing values in each column
print("\nMissing values per column:")
print(df.isnull().sum())

# Drop rows where title or text is missing
df.dropna(subset=['combined_text', 'label'], inplace=True)


In [ ]:
!pip install contractions
!pip install emoji

In [ ]:
import contractions
import emoji

def clean_text_advanced(text):
    text = str(text).lower()
    text = contractions.fix(text)  # Expand contractions
    text = emoji.replace_emoji(text, '')  # Remove emojis
    text = re.sub(r'\[.*?\]', '', text)
    text = re.sub(r'https?://\S+|www\.\S+', '', text)
    text = re.sub(r'<.*?>+', '', text)
    text = re.sub(r'[%s]' % re.escape(string.punctuation), '', text)
    text = re.sub(r'\n', ' ', text)
    text = re.sub(r'\w*\d\w*', '', text)
    tokens = text.split()
    tokens = [lemmatizer.lemmatize(word) for word in tokens if word not in stop_words and len(word) > 1]
    return " ".join(tokens)

# Apply to both columns
welfake['clean_text'] = welfake['combined_text'].apply(clean_text_advanced)
welfake = welfake.rename(columns={'clean_text': 'text'})
news['clean_text'] = news['combined_text'].apply(clean_text_advanced)
news = news.rename(columns={'clean_text': 'text'})
fakenewsnet['clean_text'] = fakenewsnet['combined_text'].apply(clean_text_advanced)
fakenewsnet = fakenewsnet.rename(columns={'clean_text': 'text'})
df = pd.concat([welfake, news, fakenewsnet])


In [ ]:

# Save individual cleaned datasets
welfake.to_csv('/content/drive/MyDrive/Mini/welfake_clean.csv', index=False)
news.to_csv('/content/drive/MyDrive/Mini/news_clean.csv', index=False)
fakenewsnet.to_csv('/content/drive/MyDrive/Mini/fakenewsnet_clean.csv', index=False)

# Save the combined dataset
df.to_csv('/content/drive/MyDrive/Mini/combined_clean.csv', index=False)

print("✅ All cleaned datasets have been saved to Google Drive (Mini folder).")

In [ ]:
# Load back the datasets from Google Drive
welfake = pd.read_csv('/content/drive/MyDrive/Mini/welfake_clean.csv')
news = pd.read_csv('/content/drive/MyDrive/Mini/news_clean.csv')
fakenewsnet = pd.read_csv('/content/drive/MyDrive/Mini/fakenewsnet_clean.csv')
df = pd.read_csv('/content/drive/MyDrive/Mini/combined_clean.csv')

# Quick check of sizes
print("welfake shape:", welfake.shape)
print("news shape:", news.shape)
print("fakenewsnet shape:", fakenewsnet.shape)
print("combined df shape:", df.shape)


In [ ]:
!pip install fasttext gensim


In [ ]:
import fasttext.util
import numpy as np

# Download pretrained FastText (Common Crawl, 2M words, 300d)
fasttext.util.download_model('en', if_exists='ignore')
ft_model_unsup = fasttext.load_model('cc.en.300.bin')

print("✅ Unsupervised FastText model loaded:", ft_model_unsup.get_dimension())

# Convert text → vector (average of word embeddings)
def text_to_fasttext_embeddings(text, ft_model):
    words = text.split()
    embeddings = [ft_model.get_word_vector(w) for w in words]
    return np.mean(embeddings, axis=0) if embeddings else np.zeros(ft_model.get_dimension())

# Create embedding matrix
X_unsup = np.vstack(df['text'].apply(lambda x: text_to_fasttext_embeddings(x, ft_model_unsup)))
y = df['label'].values

print("✅ Shape of unsupervised feature matrix:", X_unsup.shape)


In [ ]:
import pandas as pd


# Path to save training file
train_file = "/content/drive/MyDrive/Mini/fasttext_train.txt"

# Write in FastText format
with open(train_file, "w", encoding="utf-8") as f:
    for text, label in zip(df['text'], df['label']):
        f.write(f"__label__{label} {text}\n")

print("✅ Training file created at:", train_file)

# Show first few lines
!head -n 5 /content/drive/MyDrive/Mini/fasttext_train.txt


In [ ]:
import fasttext

# Prepare training file
train_file = "/content/drive/MyDrive/Mini/fasttext_train.txt"
with open(train_file, "w", encoding="utf-8") as f:
    for text, label in zip(df['text'], df['label']):
        f.write(f"__label__{label} {text}\n")

print("✅ Training file created")

# Train supervised FastText model
ft_model_sup = fasttext.train_supervised(
    input=train_file,
    lr=0.1, 
    epoch=50, 
    wordNgrams=2, 
    dim=300
)

print("✅ Supervised FastText model trained")


In [ ]:
import fasttext
import fasttext.util
import numpy as np

# ✅ Download pretrained unsupervised FastText English vectors (300d)
fasttext.util.download_model('en', if_exists='ignore')  
ft_model = fasttext.load_model('cc.en.300.bin')

print("✅ FastText model loaded:", ft_model.get_dimension(), "dimensions")

# Function to convert a text into a vector
def text_to_fasttext_embeddings(text, ft_model):
    words = text.split()
    embeddings = []
    for word in words:
        vec = ft_model.get_word_vector(word)  # Subword info handles OOV
        embeddings.append(vec)
    if embeddings:
        return np.mean(embeddings, axis=0)
    else:
        return np.zeros(ft_model.get_dimension())

# Apply to your dataset
X_fasttext = np.vstack(df['text'].apply(lambda x: text_to_fasttext_embeddings(x, ft_model)))
y = df['label'].values

print("✅ Shape of FastText feature matrix:", X_fasttext.shape)
